# 24 - Is there anything to gain at small data? (era 5x2, 10% and 3% of each hospital's training data)

Question: when each hospital holds little data (like the clients in FedGTA / OpenFGL), does knowing the twin hospitals help at all?
Same era 5x2 split as notebook 23 (5 eras x 2 hospitals). Every hospital keeps `frac` of its train and validation patients
(`regimes.subsample_frame`, as in notebook 18) and its FULL test set.
- 10% = about 470 train patients / 11,000 rows per hospital (like ogbn-arxiv per client)
- 3%  = about 140 train patients / 3,400 rows per hospital (like ogbn-products and the larger Amazon / Coauthor sets)

Arms (all end with the same 8-epoch fine-tune, local-only gets 40 epochs): **FedAvg (+ local-only)** and the **oracle**
(equal weights within the true era pair). Seeds 42, 43, 44.

**Decision rule, fixed before running:** if `oracle+ft - fedavg+ft` is below +0.005 (F1, unweighted mean over hospitals) at a
data level, there is nothing for any fingerprint to gain there. If it is at least +0.005 with the interval above zero, the next
step is to add the antibiotic-mix fingerprint and its shuffled control at that level.

Runtime: unknown (smaller data should be well under notebook 23's 1 to 1.5 h). Checkpointed after every seed; re-run cell 4 to resume.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/budget-and-era'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this branch has the notebook-21 modules
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import budget_runs as BR, budget as B, final_protocol as P
from amr_fed.partition import era_cluster_split
from amr_fed.label_fingerprints import CANDIDATES
print('imports OK | long run:', BR.LONG_ROUNDS, 'rounds, local-only', BR.LONG_LOCAL, 'epochs | Part B arms:', list(BR.PART_B_ARMS))

In [ ]:
# 3) Load cohort + check the split and the data levels
import pandas as pd
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.federated.regimes import subsample_frame
from amr_fed.topology import train_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44)
FRACS = (0.1, 0.03)
N_ERAS, PER = 5, 2
ERA_SPLIT = lambda d, s: era_cluster_split(d, n_clients=N_ERAS * PER, n_eras=N_ERAS, seed=s)
era_group = lambda raw: int(raw) // PER
raw = ERA_SPLIT(df, 42); codes, uniq = pd.factorize(raw); assign = pd.Series(codes, index=raw.index)
assert assign.nunique() == 10 and assign.index.is_unique
for frac in FRACS:
    tr = []
    for c in range(10):
        sub, code = subsample_frame(df[df[PK].map(assign) == c], 42, c, frac)
        tr.append(sub[sub[PK].map(code) == 0])
    print(f'frac {frac}: median train patients {int(pd.Series([t[PK].nunique() for t in tr]).median())}, median train rows {int(pd.Series([len(t) for t in tr]).median())}')
print('SMOKE PASS')

In [ ]:
# 4) Run FedAvg and the oracle at each data level (resume-safe)
import json, ray
os.environ.pop('AMR_RNG_PROBE', None)
RES_DIR = SRC.parent / 'results'; RES_DIR.mkdir(exist_ok=True)
ARMS = {k: dict(BR.PART_B_ARMS[k]) for k in ('fedavg', 'oracle')}
ARMS['oracle']['personalized_group_of'] = era_group      # pairs, not groups of 5
results = {}
for frac in FRACS:
    stem = f'amr_lowdata_era5x2_f{int(round(frac*100)):03d}'
    local = RES_DIR / f'{stem}.json'
    mirror = Path(f'/content/drive/MyDrive/{stem}.json') if ON_COLAB else None
    paths = [p for p in (local, mirror) if p]
    src = local if local.exists() else (mirror if mirror and mirror.exists() else None)
    res = json.load(open(src)) if src else {}
    kw = dict(seeds=SEEDS, df=df, rounds=8, local_epochs=BR.LOCAL_EPOCHS, local_only_epochs=40, hidden=128,
              partition_fn=ERA_SPLIT, n_clients=10, train_frac=frac)
    res = BR.run_part(res, ARMS, prefix=f'lowdata{frac}', kw=kw, checkpoint=lambda r, paths=paths: P.save_json(r, *paths),
                      ray_shutdown=ray.shutdown)
    P.save_json(res, *paths)
    results[frac] = res
    print(frac, 'arms done:', [a for a in res if not a.startswith('_')])

In [ ]:
# 5) Report: F1 and AUROC, paired by seed, and the pre-registered decision
import numpy as np
from amr_fed.federated.analysis import _fmt_stat, _stat
def auroc(run):
    x = run['finetune']['auroc']; return float(np.mean(x)) if isinstance(x, list) else float(x)
for frac, res in results.items():
    print(f'################ {int(frac*100)}% of each hospital\'s training data ################')
    v = BR.part_b_values(res, 'mean')
    for k, x in v.items(): print(f'  {k:<18}{x.mean():.4f} +/- {x.std():.4f}')
    for a, b in (('oracle+ft', 'fedavg+ft'), ('fedavg+ft', 'local-only (40)'), ('oracle+ft', 'local-only (40)')):
        d = v[a] - v[b]; print(_fmt_stat(f'{a} - {b}', _stat(d), d))
    A = {k: np.array([auroc(r) for r in res[k]['runs']]) for k in ('fedavg', 'oracle')}
    d = A['oracle'] - A['fedavg']
    print(f'  AUROC after fine-tune: fedavg {A["fedavg"].mean():.4f}, oracle {A["oracle"].mean():.4f}, oracle - fedavg {d.mean():+.4f}  {d.round(4)}')
    gap = (v['oracle+ft'] - v['fedavg+ft'])
    ok = gap.mean() >= 0.005 and gap.min() > 0
    print('  DECISION:', 'headroom found, add the fingerprint arms at this level' if ok else 'oracle gain below +0.005 (or not in every seed): nothing for a fingerprint to gain here')
    print()